# データベース演習 第15回

### 🔍 利用方法

- このノートブックは**閲覧専用**です。  
  自分のGoogleドライブにコピーを作成してから編集してください。

  1. メニューの「**ファイル → ドライブにコピーを保存**」を選ぶ  
  2. コピーしたノートブックの**ファイル名（左上の"xxxx"）を学籍番号に変更**してください（提出時の識別のため）

---

### 🔧 提出手順

1. Google Colab の右上にある「**共有**」ボタンをクリック  
2. 「**一般的なアクセス**」の設定を「**リンクを知っている全員**」に変更  
3. アクセス権を「**閲覧者**」に設定（⚠️「編集者」にしないこと！）  
4. 表示されるURLをコピー  
5. WebClassの提出フォームに、その**URLを貼り付けて提出**

---

### 💡 補足・注意点

- 「**編集者**」ではなく「**閲覧者**」に設定してください。  
  → 教員が間違ってファイルを上書きしてしまうのを防ぐためです。
- **提出したリンクを自分でも一度開いてみて、ちゃんと共有されているかを確認**してください。

### データベースをダウンロード

* 全データが含まれています
* Colabのランタイムが切り替わると，その度にダウンロードする必要があります
* その度にダウンロードするのが面倒な人は，自分のGoogleドライブに保存し，そのファイルを参照する方法もあります（生成AIなどで調べてみてください）．

In [ ]:
# SQLiteファイルをダウンロード

# ex_finalデータベース
!curl -sLO https://raw.githubusercontent.com/ggszk/ggszk-lab-public/refs/heads/main/db/ex_final.sqlite3


### JupySQLのインストールと有効化

In [ ]:
# Colabではjupysqlのインストールが必要
import sys
if 'google.colab' in sys.modules:
    %pip install -q jupysql

In [ ]:
# jupysqlの拡張機能を有効化
%load_ext sql

In [ ]:
# 結果表示数は以下の数値を変えれば変更できる
%config SqlMagic.displaylimit = 100 # デフォルトは10行

### ex_finalデータベース

In [ ]:
# ex_finalデータベースに接続する
%sql sqlite:///ex_final.sqlite3

### 含まれるテーブルの確認

In [ ]:
%%sql
SELECT * FROM sqlite_master;


## topic1 アプリケーション構築

### サンプルプログラム15-1

以下は，パリにいる顧客をすべて検索するプログラムである．

In [ ]:
import sqlite3

connection = sqlite3.connect('ex_final.sqlite3')

cursor = connection.cursor()
cursor.execute("SELECT * FROM customers WHERE city = 'Paris';")

for row in cursor :
    print(row)

cursor.close()
connection.close()

### サンプルプログラム15-2：パラメータの使用

以下は，変数cityに設定した都市の顧客を求めるプログラムである

* 変数cityの値を書き換えて再実行すれば，別の都市の顧客を求められる（例：city = 'London'）


In [ ]:
import sqlite3

city = 'Paris'

connection = sqlite3.connect('ex_final.sqlite3')
cursor = connection.cursor()
cursor.execute('SELECT * FROM customers WHERE city = ?;', (city,))

for row in cursor :
    print(row)
cursor.close()
connection.close()

## Topic 2: 分析SQL
### 例題1

注文詳細テーブル（order_details）には，注文に含まれる製品番号（product_idカラム），製品の単価（unit_priceカラム），製品の数（quantityカラム），その割引率（discount）が管理されている．これらを用いて，注文番号（order_idカラム），製品番号，注文と製品毎の注文金額（一つの注文における一種類の製品に関する金額総額．結果は小数第1位を四捨五入した整数とすること．整数に変換するのにround関数を用いること．カラムの別名をorder_product_priceとせよ）の3つの組を求めるSQL文を作成せよ．ただし，結果の行数は10に限定せよ．

まず，次のセルに自分でSQL文を書いて実行してみよう．解答例はその下の「例題1の解答例」セクションを開くと見られる．

In [ ]:
%%sql


#### 例題1の解答例

In [ ]:
%%sql
SELECT order_id, product_id, ROUND(unit_price*quantity*(1-discount)) AS order_product_price
FROM order_details LIMIT 10;


### 例題2

注文毎の注文金額（注文に含まれるすべての製品に関する金額総額の合計）を求めたい．注文テーブル（orders）と注文詳細テーブル（order_details）を用い，例題1の答えも参考にしつつ，注文番号，注文毎の注文金額（カラムの別名はorder_priceとせよ）の2つの組を求めるSQL文を作成せよ．ただし，結果の行数は10に限定せよ

まず，次のセルに自分でSQL文を書いて実行してみよう．解答例はその下の「例題2の解答例」セクションを開くと見られる．

In [ ]:
%%sql


#### 例題2の解答例

In [ ]:
%%sql
SELECT o.order_id, SUM(round(unit_price*quantity*(1-discount))) AS order_price
FROM orders AS o
JOIN order_details AS od ON o.order_id = od.order_id
GROUP BY o.order_id LIMIT 10;

### 例題3

注文テーブル（orders）には，注文のあった日付（order_dateカラム），顧客から要望された届け日（required_dateカラム），出荷された日付（shipped_date）を管理している．顧客から要望された届け日に間に合わなかった（出荷された日付が，顧客から要望された届け日を超えた）注文の，注文番号，顧客から要望された届け日，出荷された日付の3つの組を求めるSQL文を作成せよ．ただし，結果の行数は10に限定せよ

まず，次のセルに自分でSQL文を書いて実行してみよう．解答例はその下の「例題3の解答例」セクションを開くと見られる．

In [ ]:
%%sql


#### 例題3の解答例

解答例1：日付の大小関係を利用する

In [ ]:
%%sql
SELECT order_id , required_date, shipped_date FROM orders
WHERE required_date < shipped_date;

解答例2：ユリウス日を求める関数を使用する

In [ ]:
%%sql
SELECT order_id , required_date, shipped_date FROM orders
WHERE julianday(required_date) - julianday(shipped_date) < 0;

## 以下，最終課題

### 最終課題 問4

最終課題データベースのテーブルemployeesを用いて，ロンドン（London）に住んでいる（カラムcity）従業員の名（カラムfirst_name）と姓（カラムlast_name）を順番に出力するアプリケーションを作成せよ．出力は（カラムの）並べ替えを行わずに，Pythonのタプルをそのまま出力すること．

* 次のセルにプログラムを記述し，実行して結果を確認せよ

### 最終課題 問5

最終課題データベースのテーブルemployeesを用いて，以下のアプリケーションを作成せよ

* 検索する国の文字列を変数countryに設定し，
* 変数countryと同じ国（カラムcountry）に住む従業員を検索し，
* 従業員の名（カラムfirst_name）と姓（カラムlast_name）をこの順番で，
* 出力するアプリケーションを作成せよ．
* SQL文はパラメータ（?）を利用すること
* 出力は，Pythonのタプルをそのまま出力すること

次のセルにプログラムを記述し，変数countryの値をUSAとしたときの実行結果を残すこと

### 最終課題 問6

(1) まだ出荷されていない注文の総数を求めるSQL文を作成せよ（注：出荷されていない注文のshipped_dateはnullになっている）

In [ ]:
%%sql



(2) 顧客から要望された届け日に間に合わなかった注文の総数を求めるSQL文を作成せよ

In [ ]:
%%sql


(3) 注文番号と，注文から出荷までの日数（注文のあった日付と，出荷された日付の差とし，別名をshipping_periodとする．差を計算するときには，ユリウス日に変換してから差をとること）の組を，注文から出荷までの日数の大きい順で求めるSQL文を作成せよ．ただし，結果はカラムの並べ替えを行わずに，表示の行数は10に限定せよ（注：出荷されていない注文を除外することが必要である）

In [ ]:
%%sql


(4) 顧客毎の売上の合計を求めたい．それは，例題1で求めた「注文と製品毎の注文金額」を顧客毎に合計したものとする．顧客ID（customer_id）と顧客毎の売上の合計（別名customer_salesとする）の2つの組を求めるSQL文を作成せよ．ただし，結果はカラムの並べ替えを行わずに，表示の行数は10に限定せよ

In [ ]:
%%sql


(5) 顧客毎の発注金額の状況を調査したい．そのため，例題2で求めた注文毎の注文金額と，注文毎の注文金額をさらに顧客毎に平均をとった金額（顧客平均注文金額と呼ぶ）を並べて表示したい．注文テーブル（orders），注文詳細テーブル（order_details）を用い，注文番号（order_idカラム），顧客ID（customer_idカラム），注文毎の注文金額（別名をorder_priceとする），顧客平均注文金額（別名をcustomer_avg_priceとする．結果はround関数を用いて整数にせよ）の4つの組を求めるSQL文をウィンドウ関数を利用し作成せよ．ただし，結果はカラムの並べ替えを行わずに，表示の行数は10に限定せよ．また，カラムの表示順は次ページの検索結果例と同じとすること

In [ ]:
%%sql
